# 🔬 Low-Level Reinforcement Fine-Tuning — Qwen3-32B

> Maximum control over the RL training loop using Finetuning Low Level APIs

---

| | |
|---|---|
| **Model** | Qwen/Qwen3-32B |
| **Method** | GRPO (Generalized Reward Policy Optimization) |
| **APIs** | Finetuning Low Level APIs (Private Preview) |
| **Result** | 58.1% → **86.9%** retail_quality (+28.8pp) |

### Why Low-Level APIs?

The Foundry SDK (Phase 3) handles everything for you. But sometimes you need:
- **Custom training loops** — control over batching, rollout strategies, curriculum
- **Multi-turn agent rollouts** — the model calls tools across multiple turns during training
- **Real-time monitoring** — watch reward curves, KL divergence, and entropy live
- **Open-weight models** — train Qwen3-32B, Llama, or any model on Azure

### Table of Contents
1. [How the APIs Work](#how-it-works)
2. [Architecture](#architecture)
3. [The Training Loop (GRPO)](#training-loop)
4. [Launching a Run](#launching)
5. [Monitoring & Metrics](#monitoring)
6. [Results](#results)

---

## 1. How the Finetuning Low Level APIs Work <a id="how-it-works"></a>

The Finetuning Low Level APIs hide the complexity of distributed training behind **three primitives**:

<div style="margin:10px 0 18px;">
<table style="border-collapse:separate;border-spacing:0;width:100%;background:#0d1117;color:#c9d1d9;border:1px solid #30363d;border-radius:10px;overflow:hidden;font-family:'Inter','Segoe UI',system-ui,sans-serif;font-size:14px;">
  <thead>
    <tr style="background:#161b22;color:#58a6ff;">
      <th style="text-align:left;padding:10px 14px;border-bottom:1px solid #30363d;">Primitive</th>
      <th style="text-align:left;padding:10px 14px;border-bottom:1px solid #30363d;">What it does</th>
      <th style="text-align:left;padding:10px 14px;border-bottom:1px solid #30363d;">Where it runs</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background:rgba(188,140,255,0.10);">
      <td style="padding:10px 14px;border-left:3px solid #bc8cff;"><code style="background:#161b22;color:#bc8cff;padding:2px 8px;border-radius:6px;">client.sample()</code></td>
      <td style="padding:10px 14px;">Generate N completions from current policy</td>
      <td style="padding:10px 14px;"><span style="display:inline-flex;align-items:center;gap:6px;background:rgba(188,140,255,0.18);color:#d2a8ff;border:1px solid #bc8cff66;padding:2px 10px;border-radius:999px;font-weight:700;font-size:12px;letter-spacing:0.5px;">⚡ AZURE · GPU CLUSTER</span></td>
    </tr>
    <tr style="background:rgba(188,140,255,0.10);">
      <td style="padding:10px 14px;border-left:3px solid #bc8cff;"><code style="background:#161b22;color:#bc8cff;padding:2px 8px;border-radius:6px;">client.forward_backward()</code></td>
      <td style="padding:10px 14px;">Compute gradients via importance sampling</td>
      <td style="padding:10px 14px;"><span style="display:inline-flex;align-items:center;gap:6px;background:rgba(188,140,255,0.18);color:#d2a8ff;border:1px solid #bc8cff66;padding:2px 10px;border-radius:999px;font-weight:700;font-size:12px;letter-spacing:0.5px;">⚡ AZURE · GPU CLUSTER</span></td>
    </tr>
    <tr style="background:rgba(188,140,255,0.10);">
      <td style="padding:10px 14px;border-left:3px solid #bc8cff;"><code style="background:#161b22;color:#bc8cff;padding:2px 8px;border-radius:6px;">client.optim_step()</code></td>
      <td style="padding:10px 14px;">Update LoRA weights</td>
      <td style="padding:10px 14px;"><span style="display:inline-flex;align-items:center;gap:6px;background:rgba(188,140,255,0.18);color:#d2a8ff;border:1px solid #bc8cff66;padding:2px 10px;border-radius:999px;font-weight:700;font-size:12px;letter-spacing:0.5px;">⚡ AZURE · GPU CLUSTER</span></td>
    </tr>
  </tbody>
</table>
<div style="font-size:12px;color:#8b949e;margin-top:6px;padding-left:4px;">
  <span style="color:#bc8cff;">●</span> Highlighted rows execute on the Azure GPU cluster — your code only orchestrates these calls.
</div>
</div>

You write the **training logic** (GRPO loop, reward computation, curriculum).  
Azure handles the **infrastructure** (distributed training, GPU scheduling, checkpointing).

### The Core Loop (Pseudocode)

<pre style="background:#0d1117;color:#c9d1d9;border:1px solid #30363d;border-radius:10px;padding:14px 16px;font-family:'JetBrains Mono','Fira Code',Consolas,monospace;font-size:13px;line-height:1.55;overflow:auto;margin:10px 0;"><span style="color:#8b949e;"># ═══ CREATE SESSION — provision a LoRA adapter on Azure ═══</span>
session_id = <b style="color:#bc8cff;">client.create_session</b>(
    base_model=<span style="color:#a5d6ff;">&quot;Qwen/Qwen3-32B&quot;</span>,
    lora_config=LoRAConfig(rank=<span style="color:#79c0ff;">16</span>),
)

<span style="color:#8b949e;"># ═══ TRAINING LOOP ═══</span>
<span style="color:#ff7b72;">for</span> iteration <span style="color:#ff7b72;">in</span> range(num_iterations):
    <span style="color:#ff7b72;">for</span> prompt <span style="color:#ff7b72;">in</span> dataset:

        completions = <b style="color:#bc8cff;">client.sample</b>(                                                <span style="color:#8b949e;"># SAMPLE           — generate N completions from current policy</span>
            session_id, prompt_tokens,
            params=SamplingParams(temperature=<span style="color:#79c0ff;">1.0</span>, max_tokens=<span style="color:#79c0ff;">512</span>),
            num_samples=group_size,  <span style="color:#8b949e;"># e.g. 4 completions per prompt</span>
        )

        rewards = [grade(prompt, c) <span style="color:#ff7b72;">for</span> c <span style="color:#ff7b72;">in</span> completions]                           <span style="color:#8b949e;"># REWARD           — score each completion (your custom grader)</span>

        advantages = [r - mean(rewards) <span style="color:#ff7b72;">for</span> r <span style="color:#ff7b72;">in</span> rewards]                           <span style="color:#8b949e;"># ADVANTAGES       — GRPO: center within group</span>

        datums += build_datums(prompt, completions, advantages)                     <span style="color:#8b949e;"># BUILD DATUMS</span>

    <b style="color:#bc8cff;">client.forward_backward</b>(session_id, datums, loss_fn=<span style="color:#a5d6ff;">&quot;importance_sampling&quot;</span>)      <span style="color:#8b949e;"># FORWARD-BACKWARD — compute gradients</span>
    <b style="color:#bc8cff;">client.optim_step</b>(session_id, AdamParams(lr=<span style="color:#79c0ff;">1e-5</span>))                              <span style="color:#8b949e;"># OPTIM STEP       — update LoRA weights</span>
    client.save_weights_and_get_sampling_client(session_id)                         <span style="color:#8b949e;"># SYNC WEIGHTS     — refresh the sampler</span>


client.save_weights(session_id, <span style="color:#a5d6ff;">&quot;final&quot;</span>)                                            <span style="color:#8b949e;"># SAVE CHECKPOINT</span></pre>

---

## 2. Architecture <a id="architecture"></a>

<img src="post-training-recipe\loom\LoomHLD_13_github_dark.svg" alt="Architecture — GitHub Dark" width="760">

**Key points:**
- Your code runs locally (or in a VM) — it orchestrates the loop
- The heavy lifting (sampling, gradient computation, weight updates) runs on Azure GPU clusters
- LoRA adapters are stored server-side — you never download full model weights
- Checkpoints are saved automatically and can be resumed

---

## 3. The Retail Agent Training Loop (GRPO) <a id="training-loop"></a>

For our retail agent, the training loop does **multi-turn rollouts** — the model calls tools across multiple turns during training, just like it would in production.

<pre style="background:#0d1117;color:#c9d1d9;border:1px solid #30363d;border-radius:10px;padding:14px 16px;font-family:'JetBrains Mono','Fira Code',Consolas,monospace;font-size:13px;line-height:1.55;overflow:auto;margin:10px 0;"><span style="color:#8b949e;"># ━━━ INIT: Provision LoRA adapter on Azure ━━━</span>
session = <b style="color:#bc8cff;">client.create_session</b>(model=<span style="color:#a5d6ff;">&quot;Qwen3-32B&quot;</span>, lora=LoRAConfig(rank=<span style="color:#79c0ff;">32</span>))
sampler = get_sampling_client(session)

<span style="color:#8b949e;"># ━━━ LOOP: Iterate over scenario batches   ━━━</span>
<span style="color:#ff7b72;">for</span> batch <span style="color:#ff7b72;">in</span> dataset:
    <span style="color:#ff7b72;">for</span> scenario <span style="color:#ff7b72;">in</span> batch:                                                      <span style="color:#8b949e;"># 1. ROLLOUT — multi-turn agent interaction (client-side)</span>
        <span style="color:#ff7b72;">for</span> _ <span style="color:#ff7b72;">in</span> range(group_size):
            obs = system_prompt + tools + user_message
            <span style="color:#ff7b72;">while not</span> done:
                tokens = sampler.sample(obs)
                <span style="color:#ff7b72;">if</span> has_tool_call(tokens):
                    obs += execute_tool(tokens)                                 <span style="color:#8b949e;"># Call real tools!</span>
                <span style="color:#ff7b72;">else</span>:
                    done = <span style="color:#79c0ff;">True</span>
            reward = grade(final_response, scenario)

    advantages = rewards - mean(rewards)                                        <span style="color:#8b949e;"># 2. ADVANTAGES — GRPO baseline (no value network)</span>

    <b style="color:#bc8cff;">client.forward_backward</b>(datums, loss=<span style="color:#a5d6ff;">&quot;importance_sampling&quot;</span>)                 <span style="color:#8b949e;"># 3. TRAIN — update LoRA weights (server-side)</span>
    <b style="color:#bc8cff;">client.optim_step</b>(AdamParams(lr=<span style="color:#79c0ff;">2e-5</span>))
    sampler = <b style="color:#bc8cff;">client.sync_weights</b>()                                             <span style="color:#8b949e;"># 4. SYNC — refresh sampler with new weights</span></pre>

### Grader (Reward Signal)

Same grader as Phase 3, scoring the agent's final response:

| Dimension | Weight | What it checks |
|-----------|:------:|----------------|
| Decision Correctness | 50% | Correct action (refund/exchange/deny) + item ID |
| Financial Accuracy | 30% | Dollar amounts match ground truth |
| Format Compliance | 20% | Adheres to structured output format |

<br>
> 🔑 *"The model generates a full multi-turn interaction with real tools, then gets scored. It learns which tool sequences lead to higher rewards."*

---

## 4. Launching a Run <a id="launching"></a>

### Prerequisites
```bash
# Authentication
export AZURE_AI_API_KEY="<key>"  # or use `az login`
export PROJECT_ENDPOINT="<your Foundry project endpoint>"
```

### Start Training
```bash
bash loom_cookbook/recipes/retail_rl/launch_v4_prompt.sh
```

This creates a tmux session and starts the GRPO training loop.

### Override Hyperparameters
```bash
MAX_ITERS=25 LEARNING_RATE=2e-5 GROUP_SIZE=8 \
  bash loom_cookbook/recipes/retail_rl/launch_v4_prompt_and_graders.sh
```

### Default Configuration

| Parameter | Default | Notes |
|-----------|---------|-------|
| Model | `Qwen/Qwen3-32B` | Open-weight, 32B params |
| LoRA rank | 32 | Higher rank = more capacity |
| Learning rate | 5e-5 | With warmup |
| Group size | 16 | Completions per prompt for GRPO |
| Max iterations | 25 | ~25 gradient steps |
| Max turns | 10 | Multi-turn agent interactions |
| Loss | importance_sampling | Standard PPO-style loss |

---

## 5. Monitoring & Metrics <a id="monitoring"></a>

### Live Dashboard
```bash
python dashboard_server.py       # Opens at http://127.0.0.1:8000/
python dashboard_server.py --root ~/loom-runs --port 9000
```

Auto-discovers runs and visualizes `metrics.jsonl` in real-time (15s refresh).

### Key Metrics to Watch

| Chart | Key Metrics | What to Watch |
|-------|-------------|---------------|
| **Reward & Accuracy** | `reward/total`, `correct`, `format` | Train rising, eval stalling = overfitting |
| **KL Divergence** | `kl_sample_train_v1`, `v2` | Should stay small and flat; spikes precede reward dips |
| **Entropy & LR** | `entropy`, `lr` | Entropy falling too fast → mode collapse |
| **Group Composition** | `frac_all_good`, `frac_mixed`, `frac_all_bad` | Mixed band growing = learning happening |
| **Token Counts** | `ac_tokens_per_turn`, `ob_tokens_per_turn` | Hitting max_tokens ceiling caps reward |

### Interpreting the Dashboard

| Signal | Meaning | Action |
|--------|---------|--------|
| Train ↑ but Eval flat | Overfitting | Stop early or increase data diversity |
| Reward ↑ but Correct flat | Reward hacking | Fix grader or add constraints |
| Entropy drops fast | Mode collapse | Increase KL penalty or lower LR |
| KL spike | Instability | Reduce LR, check batch |
| `frac_all_good` dominates | Tasks too easy | Increase curriculum difficulty |
| Action tokens hitting ceiling | Truncated responses | Bump max_tokens |

---

## 6. Results <a id="results"></a>

After training completes, the fine-tuned Qwen3-32B model is deployed as `retail-qwen3-32b-finetuned`.

### Performance Comparison

| Model | retail_quality | IntentResolution | TaskCompletion | Method |
|-------|:---:|:---:|:---:|--------|
| **Qwen3-32B (RFT)** | **86.9%** | 88.5% | 62.3% | Low-level GRPO |
| o4-mini (RFT) | 82.3% | 80.6% | 40.3% | Foundry SDK RFT |
| GPT-4.1-mini (SFT) | 71.0% | 72.6% | 40.3% | SFT distillation |
| o4-mini (base) | 71.0% | 75.8% | 27.4% | — |
| GPT-5.4 (teacher) | 64.5% | 82.3% | 25.8% | — |
| Qwen3-32B (base) | 58.1% | 98.4% | 53.2% | — |

### Why Qwen3-32B + Low-Level APIs Won

| Factor | Impact |
|--------|--------|
| Multi-turn rollouts during training | Model learns tool sequences, not just final answers |
| Group size = 16 | Rich exploration → better advantage estimates |
| Real tool execution | No simulation gap — trains on actual API responses |
| Open-weight model | Full LoRA access, no API restrictions |

<br>
> 🏆 *"From 58.1% to 86.9% — the open-weight model trained with low-level APIs beats everything, including GPT-5.4, o4-mini RFT, and all SFT models."*

---

### View Full Results

Open the  <a href="https://build26evaldashboard.z20.web.core.windows.net/eval_dashboard.html" target="_blank" rel="noopener"
       style="display:inline-flex;align-items:center;gap:8px;background:#58a6ff;color:#0d1117;text-decoration:none;font-weight:700;padding:10px 18px;border-radius:10px;font-size:14px;box-shadow:0 2px 8px #58a6ff44;">
        Evaluations Dashboard
    </a>
```bash for the complete leaderboard with threshold analysis, difficulty breakdowns, and per-scenario heatmap.
```

---

### Deploy the Fine-Tuned Agent

```bash
cd deploy
azd deploy retail-qwen3-32b-finetuned
```

<!-- loom-dashboard-embed -->
---

### 🔗 Loom Run Dashboard

<div style="background:#161b22;border:1px solid #30363d;border-radius:14px;padding:18px;margin:14px 0;font-family:'Inter','Segoe UI',system-ui,sans-serif;">
  <div style="display:flex;align-items:center;justify-content:space-between;gap:14px;flex-wrap:wrap;margin-bottom:12px;">
    <div>
      <div style="font-size:18px;color:#c9d1d9;font-weight:700;margin-top:2px;">Loom Run Dashboard — <code style="color:#58a6ff;background:#0d1117;padding:2px 8px;border-radius:6px;font-size:14px;">model_afb526a9</code></div>
    </div>
    <a href="https://loom-dashboard.agreeablebeach-d151ec73.eastus.azurecontainerapps.io/#model_afb526a9" target="_blank" rel="noopener"
       style="display:inline-flex;align-items:center;gap:8px;background:#58a6ff;color:#0d1117;text-decoration:none;font-weight:700;padding:10px 18px;border-radius:10px;font-size:14px;box-shadow:0 2px 8px #58a6ff44;">
       🚀 Open in New Tab ↗
    </a>
  </div>
  <iframe src="https://loom-dashboard.agreeablebeach-d151ec73.eastus.azurecontainerapps.io/#model_afb526a9"
          width="100%" height="780"
          style="border:1px solid #30363d;border-radius:10px;background:#0d1117;"
          loading="lazy"
          referrerpolicy="no-referrer"
          sandbox="allow-scripts allow-same-origin allow-popups allow-forms"></iframe>
  <div style="margin-top:8px;font-size:12px;color:#c9d1d9;opacity:0.65;">
    If the embedded view is blocked by the host's iframe policy, use the <strong>Open in New Tab</strong> button above.
  </div>
</div>
